# E-commerce Customer Segmentation & Churn Prediction

This notebook performs RFM (Recency, Frequency, Monetary) analysis on e-commerce transaction
data, clusters customers using three different methods for comparison, and trains a
leakage-free churn prediction model.

**Note on this revision:** an earlier version of this notebook additionally built human-readable
business segment labels (e.g. "Loyal Customers", "Champions") on top of quartile-based RFM scores,
purely for stakeholder presentation. That labeling step has been removed here — it wasn't needed
for the technical pipeline, and the quartile "RFM_Score" it produced was inadvertently being fed
into the churn model, causing data leakage (see Section 6 for details). This version keeps only
the raw Recency/Frequency/Value numbers, which is both simpler and leakage-free.

In [ ]:
import datetime as dt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

import joblib

pd.set_option('display.max_columns', None)

## 1. Load & Clean Data

Dataset columns:

| Column | Description |
|---|---|
| `InvoiceNo` | Unique transaction identifier |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Number of items purchased |
| `InvoiceDate` | Date and time of the transaction |
| `UnitPrice` | Price per unit |
| `CustomerID` | Unique customer identifier |
| `Country` | Country where the transaction occurred |

In [ ]:
df = pd.read_csv("data.csv", encoding="latin-1")
print("Raw shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Missing-value check before cleaning
print(df.isnull().mean().mul(100).round(2).astype(str) + ' %')

In [ ]:
# CustomerID is essential for RFM/customer-level analysis — rows without one can't be attributed
# to any customer, so they're dropped rather than imputed.
df.dropna(subset=["CustomerID"], inplace=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")
df["TotalRevenue"] = df["Quantity"] * df["UnitPrice"]
print("Cleaned shape:", df.shape)
df.head()

## 2. Exploratory Data Analysis

A few high-level views of the data before moving into customer-level analysis.

In [ ]:
df["Country"].value_counts().head(10)

In [ ]:
top_products = df.groupby("Description")["TotalRevenue"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 10 Products by Total Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
monthly_revenue = df.set_index("InvoiceDate").resample("ME")["TotalRevenue"].sum()

plt.figure(figsize=(12, 5))
monthly_revenue.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

In [ ]:
customer_revenue = df.groupby("CustomerID")["TotalRevenue"].sum()

plt.figure(figsize=(10, 5))
sns.histplot(customer_revenue, bins=30)
plt.title("Customer Revenue Distribution")
plt.xlabel("Total Revenue per Customer")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 3. RFM Analysis (raw values only)

Recency, Frequency, and Monetary value are computed per customer directly — no quartile
scoring or business labels. These raw numbers are what feed into clustering below.

In [ ]:
reference_date = df["InvoiceDate"].max() + dt.timedelta(days=1)
print("Reference date for Recency calculation:", reference_date)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Value=("TotalRevenue", "sum"),
)
print("Number of customers:", rfm.shape[0])
rfm.head()

In [ ]:
rfm.describe()

## 4. Scaling

One scaler (`StandardScaler`) is used consistently across **all three** clustering methods
below, so the comparison between them is fair — in the previous version, K-Means used
`MinMaxScaler` while DBSCAN/Agglomerative used `StandardScaler`, which meant the three
methods weren't actually seeing the same geometry of the data.

In [ ]:
scaler = StandardScaler()
rfm_scaled = pd.DataFrame(
    scaler.fit_transform(rfm[["Recency", "Frequency", "Value"]]),
    columns=["Recency", "Frequency", "Value"],
    index=rfm.index,
)
rfm_scaled.describe()

## 5. Clustering — Three Methods Compared

### 5.1 K-Means (primary method)

Elbow method and silhouette score are used to select `k`.

In [ ]:
inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, "bx-")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
silhouette_scores = {}
for k in range(2, 10):
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(rfm_scaled)
    silhouette_scores[k] = silhouette_score(rfm_scaled, labels)

for k, score in silhouette_scores.items():
    print(f"k={k}: silhouette score = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), "bx-")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by k")
plt.show()

**Choosing k:** the highest silhouette score may not occur at the most *useful* k for the
business — a very low k (e.g. 2) can score well simply because it separates one obvious outlier
group from everyone else, without producing actionable segments. Inspect both the elbow plot and
the silhouette scores above, then set `chosen_k` below to whichever value balances statistical
fit with a workable number of segments (commonly 4 in RFM analyses — enough to distinguish
best/worst customers without fragmenting the data too far).

In [ ]:
chosen_k = 4  # adjust based on the elbow/silhouette plots above for your actual data

kmeans_final = KMeans(n_clusters=chosen_k, init="k-means++", n_init=10, random_state=42)
rfm["KMeans_Cluster"] = kmeans_final.fit_predict(rfm_scaled) + 1  # 1-indexed for readability

print(rfm["KMeans_Cluster"].value_counts().sort_index())
rfm.groupby("KMeans_Cluster")[["Recency", "Frequency", "Value"]].mean().round(1)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=rfm["Recency"], y=rfm["Value"], hue=rfm["KMeans_Cluster"], palette="viridis")
plt.title("K-Means Clusters (Recency vs. Value)")
plt.tight_layout()
plt.show()

### 5.2 DBSCAN

A K-distance plot on the same scaled data is used to pick `eps`, rather than hardcoding a value —
this makes the choice traceable to the actual data rather than a guess.

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(rfm_scaled)
distances, _ = neighbors.kneighbors(rfm_scaled)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(8, 4))
plt.plot(distances)
plt.title("K-Distance Plot (k=5)")
plt.xlabel("Points sorted by distance")
plt.ylabel("5th Nearest Neighbor Distance")
plt.show()

Inspect the plot above for the "elbow" (the point of sharpest curvature) — that distance value
is a reasonable `eps`. If the curve is flat with no clear elbow, that itself is informative: it
suggests the data is too uniformly/densely packed for density-based clustering to find meaningful
separated regions, and DBSCAN may not be a good fit for this dataset (this was the case in the
original analysis on the full ~4,300-customer dataset).

In [ ]:
chosen_eps = 0.8  # set based on the elbow in the k-distance plot above for your actual data

dbscan = DBSCAN(eps=chosen_eps, min_samples=5)
rfm["DBSCAN_Cluster"] = dbscan.fit_predict(rfm_scaled)
print(rfm["DBSCAN_Cluster"].value_counts().sort_index())
print("\n-1 indicates points DBSCAN considers noise (doesn't belong to any dense cluster).")

### 5.3 Agglomerative Clustering (+ Dendrogram)

In [ ]:
agglo = AgglomerativeClustering(n_clusters=chosen_k)
rfm["Agglo_Cluster"] = agglo.fit_predict(rfm_scaled) + 1
print(rfm["Agglo_Cluster"].value_counts().sort_index())

In [ ]:
linked = linkage(rfm_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Hierarchical Clustering Dendrogram (last 30 merges)")
plt.xlabel("Cluster size (or customer index)")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

### Save the clustered dataset

In [ ]:
rfm.to_csv("rfm_clustered.csv")
print("Saved rfm_clustered.csv —", rfm.shape[0], "customers,", rfm.shape[1], "columns.")
rfm.head()

## 6. Churn Prediction (leakage-free)

**Churn definition:** a customer is considered churned if their last purchase was more than
60 days before the reference date (`Recency > 60`).

**On feature selection — avoiding leakage:** the previous version of this notebook excluded raw
`Recency` from the model's features (correctly reasoning that using the same field the label is
derived from would be leakage), but still included `RFM_Score` — a composite score built by
*summing* quartile-binned Recency, Frequency, and Monetary scores. Since that composite score
still contains a Recency-derived component, it re-introduced much of the same leakage through
the back door. That composite scoring step has been removed entirely in this version, so the
issue can't recur — the model below is trained only on `Frequency` and `Value`, both of which are
independent of how `Churn` is defined.

In [ ]:
rfm["Churn"] = (rfm["Recency"] > 60).astype(int)
print(rfm["Churn"].value_counts())
print(f"\nChurn rate: {rfm['Churn'].mean():.1%}")

In [ ]:
X = rfm[["Frequency", "Value"]]
y = rfm["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.4f}")

**Reading the numbers above:** because the leaky `RFM_Score` feature has been removed, expect
these figures to be noticeably lower than the ~92% accuracy reported in the earlier version of
this notebook — that's the leakage being corrected, not a regression. The number that prints
above, whatever it is, is the honest one to report and to use going forward (this markdown cell
deliberately doesn't restate a number, so it can never drift out of sync with the actual output
the way the previous version's narrative text did).

In [ ]:
joblib.dump(clf, "churn_prediction_model.pkl")
print("Model saved as churn_prediction_model.pkl")
print("Model expects exactly 2 input features, in this order: ['Frequency', 'Value']")

**Important — if you have a deployed app (e.g. Streamlit) using the previous model:**
the earlier `churn_prediction_model.pkl` expected 3 features (`Frequency`, `Value`, `RFM_Score`).
This retrained model expects only 2 (`Frequency`, `Value`). Any app code that builds an input row
for prediction must be updated to match, and any UI that asks the user for/derives an `RFM_Score`
input is no longer needed for this model. Deploying this new `.pkl` without updating the app's
input construction will cause a shape-mismatch error at prediction time.

## 7. Conclusion

This notebook computes RFM metrics for each customer, compares three clustering approaches
(K-Means, DBSCAN, Agglomerative) on identically-scaled data, and trains a Random Forest churn
classifier using only features that are independent of how the churn label itself is defined.

**Changes from the previous version:**
- Removed the quartile-based RFM scoring and business-label segmentation step (was for
  stakeholder presentation only, not required for the technical pipeline)
- Fixed a data leakage issue: the churn model previously included a composite `RFM_Score` that
  indirectly encoded Recency, the same field used to define the Churn label
- Fixed inconsistent scaling across clustering methods (all three now use the same `StandardScaler`
  output)
- DBSCAN's `eps` is now chosen from an actual k-distance plot rather than hardcoded
- Removed unused imports and dead code (`k_means` import, unused `segment_labels` list)
- Metrics are now only ever printed from live code output, never restated as fixed numbers in
  markdown — so this notebook can't drift out of sync with its own results the way the previous
  version did (91% vs. 92% accuracy reported in two different places)

**Next steps:** if you have a Streamlit app or other deployment consuming
`churn_prediction_model.pkl`, update its input construction to match the new 2-feature input
(`Frequency`, `Value`) before redeploying.